# Sustainable Workload Placement in the Cloud–Edge–IoT Continuum

**Interactive student practical · about 90 minutes · work in groups**

You will run **two deterministic container workloads** at three locations (your **laptop**, an **edge-like VM**, and a **cloud VM**), compare real processing times, and combine them with **illustrative energy/network assumptions**. There are no submissions or grades: your goal is to test predictions and discuss when the best placement changes.

| What we observe | What we estimate |
|---|---|
| Computation duration, successful completion, logical bytes, checksum | Workload-related computing energy, allocated network energy, modeled application time |

**Important:** We are **not measuring electricity consumption**. Modeled values are not proof of physical energy or carbon savings. The standalone *CEI Sustainability Explorer* is a separate optional tool; you do not need it to run this notebook.

## 0. Before running · prediction (5 min)

Think about both workloads:

- **CPU computation:** A numerical job with almost no communication. Which placement might finish first? Which one might use the fewest *modeled* joules?
- **IoT aggregation:** Raw sensor readings logically originate at the **laptop** and the processed results go to the **cloud**. Would it be better to process locally, at the edge, or in the cloud?

Discuss the predictions briefly with your group. Do not worry about being right; you will compare them with the results later.

**Getting started:** Follow the included `README.md` before opening the notebook. Your instructor supplies temporary access credentials and node settings separately.

## 1. Preflight and local container images (10 min)

The notebook **and the Docker containers on your laptop** use the **external Orion-LD gateway with JWT**, publishing results through `PATCH <external-gateway>/entities/{executionId}`. The **remote edge/cloud containers** can use the direct internal Orion-LD endpoint (`PATCH <internal-url>/ngsi-ld/v1/entities/{executionId}`) without JWT, where internal access is permitted. Neither route changes the computation being measured.

The local Docker image can be built automatically. The two remote images must already be published on Docker Hub by the instructor.

In [ ]:
import os
import subprocess
from pathlib import Path
import cei_lab

candidates = [Path.cwd(), *Path.cwd().parents, *Path(cei_lab.__file__).resolve().parents]
root = next((p for p in candidates if (p / 'Dockerfile.cpu').is_file()
             and (p / 'Dockerfile.iot').is_file() and (p / 'scenarios/baseline.yaml').is_file()), None)
assert root, 'Project files not found. Extract the complete ZIP and launch Jupyter from the project folder.'
needed = ('CEI_ORION_URL', 'CEI_ORION_JWT', 'CEI_ORION_INTERNAL_URL',
          'CEI_AERIOS_URL', 'CEI_AERIOS_JWT', 'CEI_EDGE_NODE_ID',
          'CEI_CLOUD_NODE_ID', 'CEI_CPU_IMAGE_REMOTE', 'CEI_IOT_IMAGE_REMOTE',
          'CEI_GROUP_ID')
missing = [k for k in needed if not os.getenv(k) or 'REPLACE' in os.getenv(k, '') or 'YOUR_' in os.getenv(k, '')]
assert not missing, f'Missing environment configuration: {missing}. Ask the instructor.'
check = subprocess.run(['docker', 'info', '--format', '{{.ServerVersion}}'],
                       capture_output=True, text=True, timeout=20)
assert check.returncode == 0, 'Docker unavailable; check Docker Desktop/WSL integration.'
print('Lab project:', root)
print('Docker engine:', check.stdout.strip())
print('Orion/aeriOS configuration present. Tokens are intentionally hidden.')

In [ ]:
# Instructor may change to False if both local images were already built.
BUILD_LOCAL_IMAGES = True
LOCAL_CPU_IMAGE = 'cei-lab-cpu:0.2.0'
LOCAL_IOT_IMAGE = 'cei-lab-iot:0.3.0'
if BUILD_LOCAL_IMAGES:
    for dockerfile, image in [('Dockerfile.cpu', LOCAL_CPU_IMAGE),
                              ('Dockerfile.iot', LOCAL_IOT_IMAGE)]:
        print('Building:', image)
        subprocess.run(['docker', 'build', '-f', str(root/dockerfile),
                        '-t', image, str(root)], check=True)
else:
    print('Using prebuilt local Docker images; verify they are available.')

## 2. Connect the three locations

**Laptop** executes a one-shot container through Docker Desktop. **Edge / cloud** use the aeriOS REST API (one-component `man`, `isJob: true`, node ID selected in the YAML). Every run creates a unique `CEIExecution` entity in Orion-LD. The container publishes status and measurements; the notebook **polls every 1.5 seconds** until a terminal status or timeout.

We maintain a local registry of *only this group's remote service IDs*. At the end you can delete them safely using the **Clear all** section.

In [ ]:
from cei_lab.orion import OrionClient
from cei_lab.local import LocalDockerRunner
from cei_lab.aerios import AeriOSHttpClient
from cei_lab.remote import RemoteAeriOSRunner
from cei_lab.session import SessionRegistry
from cei_lab.analysis import load_scenario, observation, compare, best_feasible, export_for_explorer
from IPython.display import display, Markdown
import pandas as pd
import matplotlib.pyplot as plt

# Change this to your instructor-assigned group identifier, if any.
GROUP_ID = os.getenv('CEI_GROUP_ID', '').strip()
import re
assert re.fullmatch(r'[a-zA-Z0-9_.-]{1,64}', GROUP_ID), 'Invalid group ID: use 1-64 letters/digits/underscore/dash/dot.'
registry_file = root / 'cei_lab_state' / (GROUP_ID + '_session.json')
registry = SessionRegistry(registry_file)
orion = OrionClient.from_env()
aerios = AeriOSHttpClient.from_env()
local = LocalDockerRunner(orion, image=LOCAL_CPU_IMAGE,
                          iot_image=LOCAL_IOT_IMAGE)
remote = RemoteAeriOSRunner(
    orion_external=orion, aerios=aerios, session=registry,
    orion_internal_url=os.environ['CEI_ORION_INTERNAL_URL'],
    edge_node_id=os.environ['CEI_EDGE_NODE_ID'],
    cloud_node_id=os.environ['CEI_CLOUD_NODE_ID'],
    cpu_image=os.environ['CEI_CPU_IMAGE_REMOTE'],
    iot_image=os.environ['CEI_IOT_IMAGE_REMOTE'],
)
observations = []  # This kernel's collected data; raw measurements remain in Orion.

def store_observation(*, workload, placement, repetition, run):
    # Rerunning a cell replaces its previous measurement instead of creating
    # duplicate entries in the classroom comparison.
    item = observation(workload=workload, placement=placement,
                       repetition=repetition, run=run)
    observations[:] = [r for r in observations
                       if (r['workload'],r['placement'],r['repetition']) !=
                          (workload,placement,repetition)]
    observations.append(item)

print('Group:', GROUP_ID, '| Session:', registry.session_id)
print('Orion gateway:', orion.entities_url)
print('Remote placement nodes: configured')

## 3. Experiment A — CPU computation (18 min)

The **same prime-counting problem** runs twice on each placement. The calculation is finite and its `outputChecksum` must match everywhere. Computation time is measured **inside** the container; the longer submission-to-result time includes operational overhead and Orion polling.

**New baseline:** `limit = 2,000,000` produces a longer single-worker computation than the previous 400,000-value smoke test. **Change only if the instructor asks:** `limit` (problem size) and `workers` (parallel workers). If a run fails, tell your instructor and repeat that run before moving to calculations.

In [ ]:
CPU_LIMIT = 2_000_000
CPU_WORKERS = 1
REPETITIONS = 2
CPU_TIMEOUT_S = 180

for rep in range(1, REPETITIONS + 1):
    print(f'Laptop CPU | run {rep}/{REPETITIONS} ...', flush=True)
    run = local.run_cpu(group_id=GROUP_ID, repetition=rep,
                        limit=CPU_LIMIT, workers=CPU_WORKERS,
                        timeout_s=CPU_TIMEOUT_S)
    store_observation(workload='cpu_compute', placement='laptop', repetition=rep, run=run)
    print('  compute:', round(run.result['computeTimeMs']/1000, 3), 's',
          '| primeCount:', run.result['primeCount'])

In [ ]:
for place in ('edge', 'cloud'):
    for rep in range(1, REPETITIONS + 1):
        print(f'{place.title()} CPU | run {rep}/{REPETITIONS} | waiting for Orion result ...', flush=True)
        run = remote.run_cpu(group_id=GROUP_ID, placement=place, repetition=rep,
                             limit=CPU_LIMIT, workers=CPU_WORKERS,
                             timeout_s=CPU_TIMEOUT_S, poll_interval_s=1.5)
        store_observation(workload='cpu_compute', placement=place, repetition=rep, run=run)
        print('  completed:', round(run.result['computeTimeMs']/1000, 3), 's',
              '| primeCount:', run.result['primeCount'])

In [ ]:
cpu_records = [r for r in observations if r['workload'] == 'cpu_compute']
assert len({r['outputChecksum'] for r in cpu_records}) == 1, 'Results differ across placements!'
display(pd.DataFrame(cpu_records)[['placement','repetition','computeTimeMs',
                                   'jobWallTimeS','recordsProcessed']])
print('CPU results verified. The job wall time is NOT the modeled application time.')

## 4. Experiment B — IoT sensor aggregation (21 min)

The **same deterministic sensor readings** are reconstructed at each location from one seed. They are filtered and aggregated into much smaller window summaries. The logical data source is the **laptop**; the logical result destination is the **cloud**.

**Approach A:** Data is *not physically uploaded* from laptop to edge/cloud in these runs. Later we will estimate those transfers under an explicit network model. Dataset generation and reporting to Orion are excluded from the timed computation. The input size represents a fixed-width logical sensor record (16 bytes each); output size is the aggregated JSON-lines payload.

In [ ]:
IOT_RECORDS = 500_000
IOT_WINDOW = 500
IOT_SEED = 42
IOT_THRESHOLD = 9500
IOT_WORK_FACTOR = 45
IOT_TIMEOUT_S = 180

iot_kwargs = dict(records=IOT_RECORDS, window=IOT_WINDOW,
                  seed=IOT_SEED, threshold=IOT_THRESHOLD,
                  work_factor=IOT_WORK_FACTOR)
for rep in range(1, REPETITIONS + 1):
    print(f'Laptop IoT | run {rep}/{REPETITIONS} ...', flush=True)
    run = local.run_iot(group_id=GROUP_ID, repetition=rep,
                        timeout_s=IOT_TIMEOUT_S, **iot_kwargs)
    store_observation(workload='iot_aggregation', placement='laptop', repetition=rep, run=run)
    print('  compute:', round(run.result['computeTimeMs']/1000, 3), 's',
          '| data reduction:', round(run.result['dataReductionRatio'],1), 'x')

In [ ]:
for place in ('edge', 'cloud'):
    for rep in range(1, REPETITIONS + 1):
        print(f'{place.title()} IoT | run {rep}/{REPETITIONS} | waiting for Orion ...', flush=True)
        run = remote.run_iot(group_id=GROUP_ID, placement=place, repetition=rep,
                             timeout_s=IOT_TIMEOUT_S, poll_interval_s=1.5,
                             **iot_kwargs)
        store_observation(workload='iot_aggregation', placement=place, repetition=rep, run=run)
        print('  completed:', round(run.result['computeTimeMs']/1000,3),'s',
              '| output bytes:',run.result['outputBytes'])

In [ ]:
iot_records = [r for r in observations if r['workload']=='iot_aggregation']
assert len({r['outputChecksum'] for r in iot_records}) == 1, 'IoT results differ across placements!'
display(pd.DataFrame(iot_records)[['placement','repetition','computeTimeMs',
                                   'inputBytes','outputBytes','recordsProcessed']])
example_iot = iot_records[0]
print('Logical data reduction: %.1f x' % (example_iot['inputBytes']/example_iot['outputBytes']))
print('No measured application upload/download is claimed.')

## 5. The model: watts, joules, bits and deadlines (15 min)

These are **predefined teaching assumptions**, not electricity or network measurements. The notebook reads them from `scenarios/baseline.yaml` and shows their values below.

**Computation energy** (power × measured computation duration):

$$
\widehat E_{\rm comp,i} = P_{\rm eff,i} \times T_{\rm exec,i}
$$

**Allocated network energy** (volume × path energy intensity):

$$
\widehat E_{\rm net} = \sum_{p\;\mathrm{on\;path}} (8\,D_{p,\rm bytes})\; I_{p,\rm J/bit}
$$

**Total modeled operational energy** (joules per successful task):

$$
\widehat E_{\rm total,i} = \widehat E_{\rm comp,i} + \widehat E_{\rm net,i}
$$

**Modeled application completion time** (not a measured end-to-end data transfer):

$$
\widehat T_{\rm app,i} = T_{\rm exec,i} + \sum_p\left(\frac{8D_{p,\rm bytes}}{R_{p,\rm bit/s}} + L_{p,\rm s}\right)
$$

Where: `1 byte = 8 bits`, `1 μJ = 0.000001 J`, and `1 Mbps = 1,000,000 bits/s`.

We **neglect application data transfer** for the CPU job. For the IoT job, raw readings originate at laptop and summaries end in cloud:

- **Laptop processes:** send only *output* laptop → cloud.
- **Edge processes:** send *raw input* laptop → edge, then *output* edge → cloud.
- **Cloud processes:** send *raw input* laptop → cloud.

The model does **not** separately allocate host idle power, cooling, image download, deployment, or carbon emissions. We use one *effective workload-associated* wattage for each location.

In [ ]:
scenario = load_scenario(root / 'scenarios' / 'baseline.yaml')
ass = scenario['assumptions']
display(Markdown('### Table 1 — assumed effective compute power'))
display(pd.DataFrame([{'Location':loc, 'Effective power (W)':ass['power_w'][loc]}
                      for loc in ('laptop','edge','cloud')]))
display(Markdown('### Table 2 — assumed network paths'))
display(pd.DataFrame([{'Logical path':name.replace('_',' → '),
                       'Energy intensity (μJ/bit)':v['energy_uj_per_bit'],
                       'Effective bandwidth (Mbps)':v['bandwidth_mbps'],
                       'Fixed delay (ms)':v['fixed_latency_ms']}
                      for name,v in ass['paths'].items()]))
display(Markdown('### Table 3 — assumed application deadlines'))
display(pd.DataFrame([{'Workload':k, 'Deadline (s)':v}
                      for k,v in ass['deadline_s'].items()]))

In [ ]:
comparisons = {}
for name in ('cpu_compute','iot_aggregation'):
    comparisons[name] = compare(observations, name, scenario)
    winner = best_feasible(comparisons[name])
    display(Markdown(f"### {name.replace('_', ' ').title()} — energy and time"))
    df = pd.DataFrame(comparisons[name])[['placement','repetitions','compute_time_s',
        'effective_power_w','compute_energy_j','network_energy_j','total_energy_j',
        'modeled_app_time_s','feasible']]
    display(df.round(3))
    print('Lowest-energy feasible placement:',winner['placement'] if winner else 'none')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, workload in zip(axes, ('cpu_compute','iot_aggregation')):
    rows = comparisons[workload]
    x = [r['placement'] for r in rows]
    c = [r['compute_energy_j'] for r in rows]
    n = [r['network_energy_j'] for r in rows]
    ax.bar(x, c, label='Compute (modeled)')
    ax.bar(x, n, bottom=c, label='Network (modeled)')
    ax.set_ylabel('Estimated energy (J / completed task)')
    ax.set_title(workload.replace('_',' ').title())
    ax.legend()
plt.suptitle('Assumed power/network coefficients × real measured runtimes', y=1.04)
plt.tight_layout()
plt.show()

## 6. Placement challenge — what if conditions change? (11 min)

The lowest-energy placement might miss an application deadline. A *feasible placement* must satisfy the deadline first, then we choose the minimum modeled joules among feasible options.

Use the controls to change **one assumption at a time**. Increasing network energy intensity changes **joules**, not transfer time. Changing bandwidth changes transfer time, **not** network joules. Try tightening the deadline until the previous winner becomes infeasible.

Discuss: (1) Does the fastest location have the lowest energy? (2) How much does data reduction help? (3) Which assumption can reverse your decision?

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output

model_workload = widgets.Dropdown(options=[('IoT aggregation','iot_aggregation'),
                                         ('CPU computation','cpu_compute')],
                                  value='iot_aggregation', description='Workload:')
deadline = widgets.FloatSlider(value=15, min=1, max=90, step=1, description='Deadline (s):',
                               continuous_update=False, style={'description_width':'110px'})
network_scale = widgets.FloatSlider(value=1, min=0, max=3, step=.1,
                                    description='Net energy ×:', continuous_update=False,
                                    style={'description_width':'110px'})
bandwidth_scale = widgets.FloatSlider(value=1, min=.25, max=3, step=.05,
                                      description='Bandwidth ×:', continuous_update=False,
                                      style={'description_width':'110px'})

sensitivity_output = widgets.Output()
def redraw(change=None):
    with sensitivity_output:
        clear_output(wait=True)
        rows = compare(observations, model_workload.value, scenario,
                       deadline_s=deadline.value,
                       network_energy_multiplier=network_scale.value,
                       bandwidth_multiplier=bandwidth_scale.value)
        df = pd.DataFrame(rows)[['placement','total_energy_j','modeled_app_time_s','feasible']]
        display(df.round(3))
        best = best_feasible(rows)
        print('Lowest-energy feasible:', best['placement'] if best else 'NONE')
        print('Only model assumptions changed. Stored Orion measurements were not modified.')
for widget in (model_workload, deadline, network_scale, bandwidth_scale):
    widget.observe(redraw, names='value')
display(widgets.VBox([widgets.HBox([model_workload, deadline]),
                      widgets.HBox([network_scale, bandwidth_scale]), sensitivity_output]))
redraw()

## 7. Optional — transfer real runtimes to the standalone HTML Explorer

The HTML explorer is a **separate, offline tool**. This cell merely creates small JSON files containing execution durations (in seconds), with no JWTs or Orion connections. In the HTML page, select a matching CPU/IoT scenario, then choose **Import runtimes JSON**. Its **power and network values remain illustrative**.

In [ ]:
# Optional: uncomment to create two importable files under cei_lab_state/.
# for workload in ('cpu_compute', 'iot_aggregation'):
#     path = export_for_explorer(observations, workload,
#                               root / 'cei_lab_state' / f'{workload}_runtimes.json')
#     print('Created:', path)
print('HTML export is optional; the explorer remains independent.')

## 8. Clear all aeriOS services (end of the practical)

**Only after the runs have finished**, type `CLEAR` and click the button below. It operates on **only the service IDs in this group's local session registry**.

For each tracked service, cleanup follows the **asynchronous service lifecycle**:

1. `DELETE /hlo_fe/services/{serviceId}` starts deallocation.
2. The notebook periodically checks the corresponding **ServiceComponent `man` entity in Orion-LD** (external gateway + JWT). It waits for evidence of a **post-DELETE state transition** or a new lifecycle-status revision. Simply seeing `Finished` is *not enough*, because a one-shot job can already be `Finished` before deallocation.
3. **Only after deallocation is confirmed**, it calls `DELETE /hlo_fe/services/{serviceId}/purge` to remove the retained aeriOS service/component records.

The wait is bounded (90 seconds by default), and every phase is saved in the session registry. **If deallocation cannot be verified, purge is skipped**, so no potentially active service is forcibly purged. Click again to retry pending checks without resending accepted DELETEs. Original `CEIExecution` measurement entities stay in Orion-LD.

**Note:** The Orion lifecycle fields must be supported by the deployed aeriOS installation. If all purges remain pending, inform your instructor rather than manually purging other services.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

print("Services created in this group's registry:", len(registry.rows()))
for entry in registry.rows():
    print(entry['service_id'], entry['placement'],
          '| deletion:', entry['deletion'],
          '| deallocation:', entry.get('deallocation', 'not_checked'),
          '| purge:', entry.get('purge', 'not_requested'))

confirmation = widgets.Text(placeholder='Type CLEAR', description='Confirm:')
clear_button = widgets.Button(description='Clear all tracked services + purge',
                              button_style='danger', icon='trash')
clear_output_area = widgets.Output()

DEALLOCATION_TIMEOUT_S = 90  # bounded wait across ALL tracked services
DEALLOCATION_POLL_S = 1.5   # Orion-LD GET polling interval, seconds

def on_clear_clicked(button):
    with clear_output_area:
        clear_output(wait=True)
        if confirmation.value != 'CLEAR':
            print('No requests sent. Type CLEAR exactly.'); return
        clear_button.disabled = True
        try:
            print('Deallocating tracked services and checking Orion status before purging...', flush=True)
            reports = registry.clear_all(
                aerios, orion,
                deallocation_timeout_s=DEALLOCATION_TIMEOUT_S,
                poll_interval_s=DEALLOCATION_POLL_S,
                on_progress=lambda msg: print('  ' + msg, flush=True),
            )
            if not reports:
                print('All tracked services are already cleaned up; no requests repeated.')
            for report in reports:
                print(report)
            # A purge accepted by the older v3 prototype stays complete; its
            # historical pre-purge lifecycle cannot be retrospectively verified.
            problems = [x for x in registry.rows() if x.get('purge') != 'accepted']
            print('Tracked services still requiring cleanup:', len(problems))
            if problems:
                print('Cleanup is incomplete. Fix any reported issue and click again.')
        finally:
            clear_button.disabled = False

clear_button.on_click(on_clear_clicked)
display(widgets.VBox([confirmation, clear_button, clear_output_area]))

## 9. Discuss and finish (7 min)

With your group, explain your preferred placement for each workload and whether the deadline affected the decision. Identify **one assumption** (effective watts, network intensity, bandwidth, fixed delay, or deadline) that could reverse the result.

Remember: your timing observations are real for the available laptops/VMs, but the energy numbers are **model-based allocations**. Shared infrastructure power, idle energy, embodied impacts and data transfer were not directly measured here.

No report, submission, or grading is required. The purpose is to understand why there is no universally optimal placement across the Cloud–Edge–IoT continuum.